# Лабораторная работа 2. NumPy и Pandas

Продолжение NumPy (задачи 6–8) и основы Pandas (задачи 1–6).


In [1]:
import tempfile
from pathlib import Path
from time import perf_counter
from timeit import repeat

import numpy as np
import pandas as pd
from scipy import sparse
from threadpoolctl import threadpool_limits


## NumPy

### 6. Срезы и индексация

**6.1.** Во втором срезе 5 и 15 — индексы. Индекс 15 не входит в диапазон.


In [2]:
arr = np.arange(20)

print('С индекса 1, шаг 3:', arr[1::3])
print('Индексы 5–14 в обратном порядке:', arr[5:15][::-1])
print('Последние пять в обратном порядке:', arr[:-6:-1])


С индекса 1, шаг 3: [ 1  4  7 10 13 16 19]
Индексы 5–14 в обратном порядке: [14 13 12 11 10  9  8  7  6  5]
Последние пять в обратном порядке: [19 18 17 16 15]


**6.2.** Для шахматного узора беру клетки, у которых сумма индексов чётная.


In [3]:
rng = np.random.default_rng(42)
matrix = rng.integers(0, 10, size=(6, 6))
rows, cols = np.indices(matrix.shape)
checker = (rows + cols) % 2 == 0

print('Матрица:\n', matrix)
print('Строки 2–4, столбцы 1–3:\n', matrix[2:5, 1:4])
print('Диагональ:', np.diagonal(matrix))
print('Клетки шахматного узора:\n', np.where(checker, matrix, '.'))


Матрица:
 [[0 7 6 4 4 8]
 [0 6 2 0 5 9]
 [7 7 7 7 5 1]
 [8 4 5 3 1 9]
 [7 6 4 8 5 4]
 [4 2 0 5 8 0]]
Строки 2–4, столбцы 1–3:
 [[7 7 7]
 [4 5 3]
 [6 4 8]]
Диагональ: [0 6 7 3 5 0]
Клетки шахматного узора:
 [['0' '.' '6' '.' '4' '.']
 ['.' '6' '.' '0' '.' '9']
 ['7' '.' '7' '.' '5' '.']
 ['.' '4' '.' '3' '.' '9']
 ['7' '.' '4' '.' '5' '.']
 ['.' '2' '.' '5' '.' '0']]


**6.3.** При попарной индексации каждой строке соответствует один столбец. `np.ix_` берёт все сочетания.


In [4]:
a = np.array([10, 20, 30, 40, 50, 60])
m = np.arange(12).reshape(3, 4)

print('По индексам 0, 2, 4:', a[[0, 2, 4]])
print('Три пары индексов:', m[[0, 1, 2], [1, 3, 0]])
print('Блок 2 × 2:\n', m[np.ix_([0, 2], [1, 3])])


По индексам 0, 2, 4: [10 30 50]
Три пары индексов: [1 7 8]
Блок 2 × 2:
 [[ 1  3]
 [ 9 11]]


**6.4.** Делаю две копии массива. В одной обнуляю отрицательные числа, в другой — элементы, у которых слева и справа разные знаки. Крайние элементы не проверяю.


In [5]:
x = rng.standard_normal(1000)
nonnegative = x.copy()
nonnegative[nonnegative < 0] = 0

left, right = x[:-2], x[2:]
opposite = ((left < 0) & (right > 0)) | ((left > 0) & (right < 0))
positions = np.flatnonzero(opposite) + 1
without_crossings = x.copy()
without_crossings[positions] = 0

print('Элементов больше 1:', np.count_nonzero(x > 1))
print('После замены отрицательных:', np.round(nonnegative[:8], 3))
print('Позиций с соседями разных знаков:', positions.size)
print('После обнуления этих позиций:', np.round(without_crossings[:8], 3))


Элементов больше 1: 141
После замены отрицательных: [0.878 0.    0.    0.    1.223 0.    0.    0.   ]
Позиций с соседями разных знаков: 500
После обнуления этих позиций: [ 0.878  0.    -0.185  0.     1.223  0.    -0.428  0.   ]


### 7. Файлы и скорость обхода массива

**7.1–7.2.** Записываю `big.dat` и читаю его блоками по 1000 строк. Для среднего накапливаю сумму в `float64`.


In [6]:
shape = (50_000, 200)
block_rows = 1000
dtype = np.dtype('float32')

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / 'big.dat'
    mapped = np.memmap(path, mode='w+', dtype=dtype, shape=shape)
    for start in range(0, shape[0], block_rows):
        mapped[start:start + block_rows] = rng.random((block_rows, shape[1]), dtype=np.float32)
    mapped.flush()
    del mapped

    size = path.stat().st_size
    assert size == shape[0] * shape[1] * dtype.itemsize == 40_000_000
    mapped = np.memmap(path, mode='r', dtype=dtype, shape=shape)
    minimum, maximum = np.inf, -np.inf
    total, count = 0.0, 0
    for start in range(0, shape[0], block_rows):
        block = mapped[start:start + block_rows]
        minimum = min(minimum, float(block.min()))
        maximum = max(maximum, float(block.max()))
        total += float(block.sum(dtype=np.float64))
        count += block.size
    del block, mapped

print(f'Размер: {size:,} байт')
print(f'min = {minimum:.8f}, max = {maximum:.8f}, mean = {total / count:.8f}')


Размер: 40,000,000 байт
min = 0.00000000, max = 0.99999988, mean = 0.49993806


**7.3.** Массивы C и F содержат одинаковые значения, но отличаются порядком хранения. Для каждого варианта отдельно замеряю обход двумя вложенными циклами.


In [7]:
def sum_rows(arr):
    total = 0.0
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            total += arr[i, j]
    return total


def sum_columns(arr):
    total = 0.0
    for j in range(arr.shape[1]):
        for i in range(arr.shape[0]):
            total += arr[i, j]
    return total


c_order = rng.random((3000, 3000))
f_order = np.asfortranarray(c_order)
expected_sum = c_order.sum()

for order, arr in [('C', c_order), ('F', f_order)]:
    for label, sum_values in [('строки', sum_rows), ('столбцы', sum_columns)]:
        start = perf_counter()
        total = sum_values(arr)
        elapsed = perf_counter() - start
        assert np.isclose(total, expected_sum, rtol=1e-11, atol=0)
        print(f'{order}-order, {label:8}: {elapsed:.3f} с')

del arr, c_order, f_order


C-order, строки  : 1.098 с
C-order, столбцы : 1.975 с
F-order, строки  : 2.521 с
F-order, столбцы : 2.403 с


В C-order соседние элементы лежат рядом по строкам, в F-order — по столбцам. Последовательный доступ обычно быстрее, но на замер влияют и накладные расходы Python-циклов.

**7.4.** Вычисляю `(a + b) / (a - b + 1)` тремя способами. Беру значения от 0 до 0.5, чтобы знаменатель не приближался к нулю. Создание исходных списков и выделение буферов выполняются до замера.


In [8]:
def calculate_python(left, right):
    result = []
    for a_value, b_value in zip(left, right):
        result.append((a_value + b_value) / (a_value - b_value + 1))
    return result


n = 10_000_000
a = rng.random(n) * 0.5
b = rng.random(n) * 0.5
left, right = a.tolist(), b.tolist()

start = perf_counter()
python_values = calculate_python(left, right)
python_time = perf_counter() - start
del left, right
python_values = np.asarray(python_values)

start = perf_counter()
numpy_values = (a + b) / (a - b + 1)
numpy_time = perf_counter() - start

numerator = np.empty_like(a)
denominator = np.empty_like(a)
start = perf_counter()
np.add(a, b, out=numerator)
np.subtract(a, b, out=denominator)
np.add(denominator, 1, out=denominator)
np.divide(numerator, denominator, out=numerator)
buffer_time = perf_counter() - start

np.testing.assert_allclose(numpy_values, python_values, rtol=1e-13, atol=0)
np.testing.assert_allclose(numerator, python_values, rtol=1e-13, atol=0)
print(f'Python: {python_time:.3f} с')
print(f'NumPy в одну строку: {numpy_time:.3f} с')
print(f'NumPy с out=: {buffer_time:.3f} с')
del a, b, python_values, numpy_values, numerator, denominator


Python: 2.517 с
NumPy в одну строку: 0.179 с
NumPy с out=: 0.127 с


`out=` позволяет переиспользовать два буфера. Разница по времени зависит от компьютера и состояния памяти в момент запуска.

### 8. Разреженные матрицы

**8.1.** В материалах нет самих списков `row`, `col`, `data`, поэтому создаю их и добавляю пару совпадающих координат. В CSR значения с такими координатами складываются.


In [9]:
row = rng.integers(0, 5000, size=10_000).tolist()
col = rng.integers(0, 5000, size=10_000).tolist()
data = rng.integers(1, 10, size=10_000).tolist()
row[1], col[1] = row[0], col[0]

coo = sparse.coo_matrix((data, (row, col)), shape=(5000, 5000))
csr = coo.tocsr()
unique = len(set(zip(row, col)))
assert coo.nnz == 10_000
assert csr.nnz == unique

first_position_sum = sum(value for r, c, value in zip(row, col, data)
                         if (r, c) == (row[0], col[0]))
assert csr[row[0], col[0]] == first_position_sum
print(f'COO: {coo.nnz}, CSR: {csr.nnz}, уникальных координат: {unique}')


COO: 10000, CSR: 9998, уникальных координат: 9998


**8.2.** Сравниваю одну и ту же матрицу в CSR и dense. Для dense ограничиваю BLAS одним потоком и беру медиану пяти серий замеров.


In [10]:
def compare_matvec(csr, vector):
    dense = csr.toarray()
    with threadpool_limits(limits=1, user_api='blas'):
        np.testing.assert_allclose(csr @ vector, dense @ vector, rtol=1e-12, atol=1e-12)
        csr_time = np.median(repeat(lambda: csr @ vector, number=20, repeat=5)) / 20
        dense_time = np.median(repeat(lambda: dense @ vector, number=20, repeat=5)) / 20
    csr_bytes = csr.data.nbytes + csr.indices.nbytes + csr.indptr.nbytes
    return csr_bytes, dense.nbytes, csr_time, dense_time


csr = sparse.random(4000, 4000, density=0.005, format='csr', dtype=np.float64, rng=rng)
csr_bytes, dense_bytes, csr_time, dense_time = compare_matvec(csr, rng.random(4000))
print(f'CSR:   {csr_bytes:,} байт, {csr_time * 1000:.4f} мс')
print(f'Dense: {dense_bytes:,} байт, {dense_time * 1000:.4f} мс')


CSR:   976,004 байт, 0.1978 мс
Dense: 128,000,000 байт, 6.6312 мс


**8.3.** Меняю плотность и снова замеряю `A @ x`. В сохранённом запуске переход получился между 0.2 и 0.5.


In [11]:
densities = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
vector = rng.random(2000)
measurements = []

for density in densities:
    csr = sparse.random(2000, 2000, density=density, format='csr', dtype=np.float64, rng=rng)
    _, _, csr_time, dense_time = compare_matvec(csr, vector)
    measurements.append((density, csr_time * 1000, dense_time * 1000))

timings = pd.DataFrame(measurements, columns=['Плотность', 'CSR, мс', 'Dense, мс'])
print(timings.to_string(index=False, float_format='{:.4f}'.format))

first_dense_win = timings.loc[timings['CSR, мс'] >= timings['Dense, мс'], 'Плотность']
if first_dense_win.empty:
    print('CSR быстрее при всех измеренных плотностях.')
else:
    crossover = first_dense_win.iloc[0]
    previous = timings.loc[timings['Плотность'] < crossover, 'Плотность']
    print(f'Первая проверенная плотность, где CSR не быстрее: {crossover:g}')
    if not previous.empty:
        print(f'Переход по этим замерам между {previous.iloc[-1]:g} и {crossover:g}.')


 Плотность  CSR, мс  Dense, мс
    0.0010   0.0128     1.5809
    0.0050   0.0345     1.4852
    0.0100   0.0483     1.7560
    0.0500   0.1727     1.7714
    0.1000   0.4206     2.1590
    0.2000   0.8050     1.6484
    0.5000   2.2884     1.5993
    1.0000   6.5915     2.3248
Первая проверенная плотность, где CSR не быстрее: 0.5
Переход по этим замерам между 0.2 и 0.5.


## Pandas

### 1. Series и DataFrame

Таблица сотрудников вымышленная. Оставил несколько пропусков, в том числе отдел без единой известной зарплаты.


In [12]:
series = pd.Series([12, 17, 9], index=['a', 'b', 'c'])
df = pd.DataFrame({
    'name': ['Анна', 'Борис', 'Вера', 'Глеб', 'Дарья', 'Илья', 'Лена', 'Миша'],
    'department': ['IT', 'IT', 'IT', 'Sales', 'Sales', 'HR', 'HR', 'Support'],
    'salary': [92_000, 76_000, None, 68_000, 72_000, 65_000, None, None],
    'bonus': [10_000, None, 8_000, 5_000, None, 3_000, None, 2_000],
    'hire_date': pd.to_datetime([
        '2022-02-14', '2023-04-03', '2024-01-11', '2021-06-21',
        '2023-03-15', '2022-09-26', '2024-06-10', '2024-10-02',
    ]),
})

print(series.to_string())
print('Первые строки:\n', df.head().to_string(index=False))
print('Последние строки:\n', df.tail().to_string(index=False))
df.info()
print('Статистика:\n', df[['salary', 'bonus']].describe().round(2).to_string())


a    12
b    17
c     9
Первые строки:
  name department  salary   bonus  hire_date
 Анна         IT 92000.0 10000.0 2022-02-14
Борис         IT 76000.0     NaN 2023-04-03
 Вера         IT     NaN  8000.0 2024-01-11
 Глеб      Sales 68000.0  5000.0 2021-06-21
Дарья      Sales 72000.0     NaN 2023-03-15
Последние строки:
  name department  salary  bonus  hire_date
 Глеб      Sales 68000.0 5000.0 2021-06-21
Дарья      Sales 72000.0    NaN 2023-03-15
 Илья         HR 65000.0 3000.0 2022-09-26
 Лена         HR     NaN    NaN 2024-06-10
 Миша    Support     NaN 2000.0 2024-10-02
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   name        8 non-null      object        
 1   department  8 non-null      object        
 2   salary      5 non-null      float64       
 3   bonus       5 non-null      float64       
 4   hire_date   8 non-null    

### 2. Выборка и фильтрация


In [13]:
print('Сотрудники IT:\n', df.loc[df['department'] == 'IT', ['name', 'salary']].to_string(index=False))
print('Первые 3 строки и 2 столбца:\n', df.iloc[:3, :2].to_string(index=False))
print('Зарплата выше 70 000:\n', df.loc[df['salary'] > 70_000, ['name', 'salary']].to_string(index=False))

df['total_comp'] = df['salary'] + df['bonus'].fillna(0)
df = df.drop(columns='hire_date')
print(df.to_string(index=False))


Сотрудники IT:
  name  salary
 Анна 92000.0
Борис 76000.0
 Вера     NaN
Первые 3 строки и 2 столбца:
  name department
 Анна         IT
Борис         IT
 Вера         IT
Зарплата выше 70 000:
  name  salary
 Анна 92000.0
Борис 76000.0
Дарья 72000.0
 name department  salary   bonus  total_comp
 Анна         IT 92000.0 10000.0    102000.0
Борис         IT 76000.0     NaN     76000.0
 Вера         IT     NaN  8000.0         NaN
 Глеб      Sales 68000.0  5000.0     73000.0
Дарья      Sales 72000.0     NaN     72000.0
 Илья         HR 65000.0  3000.0     68000.0
 Лена         HR     NaN     NaN         NaN
 Миша    Support     NaN  2000.0         NaN


### 3. Пропущенные значения

Среднее по отделу заполняет зарплаты Веры и Лены. Для Support среднее посчитать нельзя, поэтому строка с Мишей удаляется. После заполнения пересчитываю `total_comp`.


In [14]:
print('Пропуски до обработки:\n', df.isna().sum().to_string())

department_mean = df.groupby('department')['salary'].transform('mean')
df['salary'] = df['salary'].fillna(department_mean)
df['bonus'] = df['bonus'].fillna(0)
df = df.dropna(subset=['salary']).copy()
df['total_comp'] = df['salary'] + df['bonus']
print(df.to_string(index=False))


Пропуски до обработки:
 name          0
department    0
salary        3
bonus         3
total_comp    3
 name department  salary   bonus  total_comp
 Анна         IT 92000.0 10000.0    102000.0
Борис         IT 76000.0     0.0     76000.0
 Вера         IT 84000.0  8000.0     92000.0
 Глеб      Sales 68000.0  5000.0     73000.0
Дарья      Sales 72000.0     0.0     72000.0
 Илья         HR 65000.0  3000.0     68000.0
 Лена         HR 65000.0     0.0     65000.0


### 4. Группировка и сводная таблица


In [15]:
mean_salary = df.groupby('department')['salary'].mean()
summary = df.groupby('department').agg(
    mean_salary=('salary', 'mean'),
    max_bonus=('bonus', 'max'),
    employees=('name', 'size'),
)
pivot = df.pivot_table(index='department', values='salary', aggfunc='mean')

print('Средняя зарплата:\n', mean_salary.to_string())
print('Сводка по отделам:\n', summary.to_string())
print('Сводная таблица:\n', pivot.to_string())
print('По убыванию средней зарплаты:\n', mean_salary.sort_values(ascending=False).to_string())


Средняя зарплата:
 department
HR       65000.0
IT       84000.0
Sales    70000.0
Сводка по отделам:
             mean_salary  max_bonus  employees
department                                   
HR              65000.0     3000.0          2
IT              84000.0    10000.0          3
Sales           70000.0     5000.0          2
Сводная таблица:
              salary
department         
HR          65000.0
IT          84000.0
Sales       70000.0
По убыванию средней зарплаты:
 department
IT       84000.0
Sales    70000.0
HR       65000.0


### 5. Объединение таблиц

Через `left join` остаются все сотрудники, в том числе из HR, где руководитель в справочнике не указан. Затем добавляю нового сотрудника IT и сбрасываю индекс.


In [16]:
managers = pd.DataFrame({'department': ['IT', 'Sales'], 'manager': ['Олег', 'Мария']})
df = df.merge(managers, on='department', how='left', validate='many_to_one')

new_employee = pd.DataFrame({
    'name': ['Саша'], 'department': ['IT'], 'salary': [80_000], 'bonus': [5_000],
})
new_employee['total_comp'] = new_employee['salary'] + new_employee['bonus']
new_employee = new_employee.merge(managers, on='department', how='left', validate='many_to_one')

df = pd.concat([df, new_employee]).reset_index(drop=True)
print(df.to_string())


    name department   salary    bonus  total_comp manager
0   Анна         IT  92000.0  10000.0    102000.0    Олег
1  Борис         IT  76000.0      0.0     76000.0    Олег
2   Вера         IT  84000.0   8000.0     92000.0    Олег
3   Глеб      Sales  68000.0   5000.0     73000.0   Мария
4  Дарья      Sales  72000.0      0.0     72000.0   Мария
5   Илья         HR  65000.0   3000.0     68000.0     NaN
6   Лена         HR  65000.0      0.0     65000.0     NaN
7   Саша         IT  80000.0   5000.0     85000.0    Олег


### 6. Мини-анализ

В итоговой таблице уже есть новый сотрудник. Доля — это среднее по маске `salary > 75000` для каждого отдела.


In [17]:
top3 = df.nlargest(3, 'total_comp')[['name', 'department', 'total_comp']]
department_averages = df.groupby('department')[['salary', 'bonus']].mean().round(2)
above_threshold = df['salary'].gt(75_000).groupby(df['department']).mean()

print('Топ-3 по компенсации:\n', top3.to_string(index=False))
print('Средняя зарплата и бонус:\n', department_averages.to_string())
print('Доля с зарплатой выше 75 000:\n', above_threshold.map('{:.1%}'.format).to_string())


Топ-3 по компенсации:
 name department  total_comp
Анна         IT    102000.0
Вера         IT     92000.0
Саша         IT     85000.0
Средняя зарплата и бонус:
              salary   bonus
department                 
HR          65000.0  1500.0
IT          83000.0  5750.0
Sales       70000.0  2500.0
Доля с зарплатой выше 75 000:
 department
HR         0.0%
IT       100.0%
Sales      0.0%
